In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import lamindb as ln
import time
import numpy as np
import celltypist

ln.track("osfd1cmuw1S8")

In [ ]:
CLUSTER_KEY    = 'leiden_res1.10'
ANNOTATION_CSV = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data/fh1_pretx_deg_leiden_res1.10_wcn_FINAL_MAH_1.xlsx - Unified_Manual_Labels.csv'
DATA_DIR       = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data/annotation'
FIGURES_DIR    = '/home/workspace/PCD_data/xenium/fh1_pretreatment/figures/annotation'

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
annotation_label = ln.ULabel(name='annotation').save()
annotated_label  = ln.ULabel(name='annotated').save()
manual_label     = ln.ULabel(name='manual').save()

ln.Param(name='annotation_source', dtype=str).save()
ln.Param(name='n_labels',          dtype=int).save()

# Manual labeling

In [ ]:
ln.Artifact.search("fh1_pretreatment_combined_clustered").to_dataframe()

In [ ]:
combined = ln.Artifact.get('9FrxD0fbe2MT8PMj0004').load()

In [ ]:
labels    = pd.read_csv(ANNOTATION_CSV)

obs_clusters   = set(combined.obs[CLUSTER_KEY].astype(int))
label_clusters = set(labels['Cluster'])

In [ ]:
cluster_ids = combined.obs[CLUSTER_KEY].astype(int)

# adds obs['cell_type_unified'] (CSV 'Unified label' per cluster); .X, .raw and all other obs columns unchanged; in memory only
combined.obs['cell_type_unified'] = cluster_ids.map(labels.set_index('Cluster')['Unified label']).astype('category')
# adds obs['cell_type_short'] (CSV 'Short' per cluster); .X, .raw and all other obs columns unchanged; in memory only
combined.obs['cell_type_short'] = cluster_ids.map(labels.set_index('Cluster')['Short']).astype('category')

In [ ]:
sc.pl.umap(combined, color='cell_type_short', size=1, legend_loc='right margin', title='FH1 pretreatment annotation')

In [ ]:
COUNTS_KEY = 'fh1_pretreatment_annotation_counts_per_sample.csv'

counts_per_sample = pd.crosstab(combined.obs['cell_type_short'], combined.obs['sample'])
counts_path = os.path.join(DATA_DIR, COUNTS_KEY)
counts_per_sample.to_csv(counts_path)  # writes cell counts per label and sample to DATA_DIR
counts_artifact = ln.Artifact(counts_path, key=COUNTS_KEY,  # new lamindb artifact (key COUNTS_KEY)
                              description='FH1 pretreatment cell counts per annotation label and sample').save()
counts_artifact.ulabels.add(annotation_label)

counts_per_sample

In [ ]:
# Writes a new lamindb artifact (key LABELS_KEY) holding the full CSV incl. the long text columns; the source CSV is not modified.
labels_artifact = ln.Artifact(ANNOTATION_CSV, key='fh1_pretreatment_cluster_annotations_res1.10.csv',
                              description='FH1 pretreatment manual cluster annotations at leiden res 1.10 (labels, comments, marker evidence)').save()
labels_artifact.ulabels.add(annotation_label, manual_label)
labels_artifact.features.add_values({'clustering_key': CLUSTER_KEY, 'n_clusters': len(labels)})

In [ ]:
# Writes a new lamindb artifact (key ANNOTATED_KEY); the clustered input artifact is not touched.
annotated_artifact = ln.Artifact.from_anndata(combined, key='fh1_pretreatment_combined_annotated.h5ad',
                                              description='Combined FH1 pretreatment Xenium object with manual cluster annotations (leiden res 1.10)').save()
annotated_artifact.ulabels.add(annotation_label, annotated_label, manual_label)
annotated_artifact.features.add_values({
    'clustering_key':    CLUSTER_KEY,
    'n_clusters':        len(labels),
    'n_labels':          combined.obs['cell_type_unified'].nunique(),
    'annotation_source': os.path.basename(ANNOTATION_CSV),
})

# TissDiss label transfer

In [ ]:
# # combined = ln.Artifact.get('9FrxD0fbe2MT8PMj0004').load()
# tissdiss = sc.read_h5ad('/home/workspace/spatial-pcd-data-release/files/tissdiss_object.h5ad')

In [ ]:
# tissdiss.obs.annotation.value_counts()

In [ ]:
# xenium_genes = combined.var_names
# shared_genes = xenium_genes[xenium_genes.isin(tissdiss.var_names)]

# tissdiss_panel = tissdiss.raw.to_adata()[:, shared_genes].copy()
# sc.pp.normalize_total(tissdiss_panel, target_sum=1e4)
# sc.pp.log1p(tissdiss_panel)

In [ ]:
# downsample_index = celltypist.samples.downsample_adata(tissdiss_panel, mode = 'each', n_cells = 6000, by = 'annotation', return_index = True)

In [ ]:
# t_start = time.time()
# model_fs = celltypist.train(tissdiss_panel[downsample_index], 'annotation', n_jobs = 24, max_iter = 300, use_SGD = True)
# t_end = time.time()
# print(f"Time elapsed: {t_end - t_start} seconds")

In [ ]:
# gene_index = np.argpartition(np.abs(model_fs.classifier.coef_), -100, axis = 1)[:, -100:]
# gene_index = np.unique(gene_index)
# print(f"Number of genes selected: {len(gene_index)}")

In [ ]:
# # Add `check_expression = False` to bypass expression check with only a subset of genes.
# t_start = time.time()
# model = celltypist.train(tissdiss_panel[downsample_index, gene_index], 'annotation', check_expression = False, n_jobs = 24, max_iter = 200)
# t_end = time.time()
# print(f"Time elapsed: {(t_end - t_start)/60} minutes")

In [ ]:
# model.write('/home/workspace/PCD_data/xenium/fh1_pretreatment/data/tissdiss_celltypist_model.pkl')

In [ ]:
combined = ln.Artifact.get(uid='XxoCfANlpvEhnlSP0000').load()

In [ ]:
# CellTypist prediction without over-clustering and majority-voting.
sc.pp.normalize_total(combined, target_sum=1e4)
sc.pp.log1p(combined)
combined.obs_names_make_unique()

t_start = time.time()
predictions = celltypist.annotate(combined, model = '/home/workspace/PCD_data/xenium/fh1_pretreatment/data/tissdiss_celltypist_model.pkl')
t_end = time.time()
print(f"Time elapsed: {t_end - t_start} seconds")

In [ ]:
predictions

In [ ]:
# You can also change the value of `use_as_prediction` to `predicted_labels` to compare the raw prediction result with the pre-defined cell types.
combined.obs_names_make_unique()
celltypist.dotplot(predictions, use_as_reference = 'cell_type_unified', use_as_prediction = 'predicted_labels')

In [ ]:
adata = predictions.to_adata()

In [ ]:
adata.obs.iloc[:, -4:]

In [ ]:
sc.pl.umap(adata, color = ['cell_type_short', 'predicted_labels'], legend_loc = 'on data')

In [ ]:
# ln.finish()